**Construcción capa Gold**

Importamos librerias

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

CATALOG      = "proyecto_smart_claims"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

spark = SparkSession.builder.getOrCreate()

Leemos las tablas Silver

In [0]:
customers_clean  = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers_clean")
policies_clean   = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.policies_clean")
claims_clean     = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.claims_clean")
telematics_clean = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.telematics_clean")

Agregar telemática por vehiculo

**Tabla: gold.aggregated_telematics**

In [0]:
aggregated_telematics = (
    telematics_clean
    .groupBy("chassis_no")
    .agg(
        F.round(F.avg("speed"),     2).alias("avg_speed"),
        F.round(F.max("speed"),     2).alias("max_speed"),
        F.round(F.min("speed"),     2).alias("min_speed"),
        F.round(F.avg("latitude"),  6).alias("avg_latitude"),
        F.round(F.avg("longitude"), 6).alias("avg_longitude"),
        F.count("*").alias("total_events"),
        F.min("event_timestamp").alias("first_event"),
        F.max("event_timestamp").alias("last_event"),
    )
)

aggregated_telematics.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.aggregated_telematics")

print("✅ gold.aggregated_telematics creada")
aggregated_telematics.show(5)

Unir reclamos, pólizas y clientes

**Tabla: gold.customer_claim_policy**

In [0]:
customer_claim_policy = (
    claims_clean
    # Join con policies por policy_no
    .join(
        policies_clean,
        claims_clean["policy_no"] == policies_clean["POLICY_NO"],
        how="inner"
    )
    # Join con customers por customer id
    .join(
        customers_clean,
        policies_clean["CUST_ID"] == customers_clean["customer_id"],
        how="left"
    )
    .select(
        # Campos del reclamo
        claims_clean["claim_no"],
        claims_clean["policy_no"],
        claims_clean["claim_date"],
        claims_clean["total"].alias("claim_total"),
        claims_clean["age"],

        # Campos de la póliza
        policies_clean["CHASSIS_NO"].alias("chassis_no"),
        policies_clean["PREMIUM"].alias("premium"),
        policies_clean["POL_EFF_DATE"].alias("policy_start_date"),
        policies_clean["POL_EXPIRY_DATE"].alias("policy_end_date"),
        policies_clean["CUST_ID"].alias("cust_id"),

        # Campos del cliente
        customers_clean["first_name"],
        customers_clean["last_name"],
        customers_clean["date_of_birth"],
        customers_clean["address"],
    )
)

customer_claim_policy.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy")

print("✅ gold.customer_claim_policy creada")
customer_claim_policy.show(5, truncate=False)

Enriquecer con telemática agregada

**Tabla: gold.customer_claim_policy_telematics**

In [0]:
customer_claim_policy_telematics = (
    customer_claim_policy
    .join(
        aggregated_telematics,
        on="chassis_no",
        how="left"
    )
)

customer_claim_policy_telematics.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy_telematics")

print("✅ gold.customer_claim_policy_telematics creada")
customer_claim_policy_telematics.show(5, truncate=False)

**Resumen Final**

In [0]:
print("=" * 60)
print("RESUMEN FINAL - CONTEO DE FILAS POR TABLA GOLD")
print("=" * 60)

tablas_gold = {
    "aggregated_telematics":            aggregated_telematics,
    "customer_claim_policy":            customer_claim_policy,
    "customer_claim_policy_telematics": customer_claim_policy_telematics,
}

for nombre, df in tablas_gold.items():
    print(f"  gold.{nombre}: {df.count():,} filas")

**VERIFICACIÓN**

Verificar tablas

In [0]:
%sql
SHOW TABLES IN proyecto_smart_claims.gold;

Verificar que cada reclamo tiene póliza

In [0]:
spark.sql(f"""
SELECT *
FROM {CATALOG}.{GOLD_SCHEMA}.customer_claim_policy
WHERE policy_no IS NULL
""").show()

Verificar que cada reclamo tiene cliente

In [0]:
spark.sql(f"""
SELECT *
FROM {CATALOG}.{GOLD_SCHEMA}.customer_claim_policy
WHERE cust_id IS NULL
""").show()

Validar unión con telemática

In [0]:
spark.sql(f"""
SELECT *
FROM {CATALOG}.{GOLD_SCHEMA}.customer_claim_policy_telematics
WHERE avg_speed IS NULL
""").show()

Validar integridad del join principal

In [0]:
spark.sql(f"""
SELECT COUNT(*) as total_claims
FROM {CATALOG}.{SILVER_SCHEMA}.claims_clean
""").show()

spark.sql(f"""
SELECT COUNT(*) as total_gold
FROM {CATALOG}.{GOLD_SCHEMA}.customer_claim_policy
""").show()

Validación de negocio

In [0]:
display(
    spark.sql(f"""
    SELECT 
        claim_no,
        first_name,
        last_name,
        policy_no,
        premium,
        avg_speed,
        claim_total
    FROM {CATALOG}.{GOLD_SCHEMA}.customer_claim_policy_telematics
    LIMIT 10
    """)
)